# MVC Project — Multilayer Perceptron (MNIST)
**Task 7: Python Implementation**  
All forward pass, loss, backpropagation, and weight-update code uses **NumPy only**.  
Architecture mirrors Tasks 1–6 (scaled to MNIST): 784 → 128 → 64 → 10

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, urllib.request

np.random.seed(42)

## Step 1 — Sigmoid Activation Function (Task 1)

In [ ]:
def sigmoid(z):
    """Sigmoid activation function with clipping to prevent overflow."""
    z = np.clip(z, -500, 500)  # prevent exp overflow
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_derivative(a):
    """Derivative of sigmoid expressed in terms of the activation a.
    Derived in Task 3: sigma'(z) = sigma(z)*(1 - sigma(z)) = a*(1-a)
    """
    return a * (1.0 - a)

# Quick sanity check
print(f"sigmoid(0) = {sigmoid(0):.4f}  (expected 0.5000)")
print(f"sigmoid_derivative(0.5) = {sigmoid_derivative(0.5):.4f}  (expected 0.2500)")

## Step 2 — Forward Pass (Task 1)

In [ ]:
def forward_pass(X, W1, b1, W2, b2, W3, b3):
    """
    Forward pass through the 3-layer MLP.
    Stores all intermediate values needed for backpropagation.
    
    Args:
        X:  input matrix (m, 784)
        W1: weights layer 1 (784, 128)
        b1: biases  layer 1 (1, 128)
        W2: weights layer 2 (128, 64)
        b2: biases  layer 2 (1, 64)
        W3: weights layer 3 (64, 10)
        b3: biases  layer 3 (1, 10)
    Returns:
        Z1, A1, Z2, A2, Z3, A3
    """
    # Layer 1
    Z1 = X @ W1 + b1          # (m, 128)
    A1 = sigmoid(Z1)           # (m, 128)

    # Layer 2
    Z2 = A1 @ W2 + b2          # (m, 64)
    A2 = sigmoid(Z2)           # (m, 64)

    # Output layer
    Z3 = A2 @ W3 + b3          # (m, 10)
    A3 = sigmoid(Z3)           # (m, 10) — probabilities

    return Z1, A1, Z2, A2, Z3, A3

## Step 3 — MSE Loss (Task 2)

In [ ]:
def mse_loss(Y_true, Y_pred):
    """
    Mean Squared Error loss.
    L = (1/m) * sum((Y_true - Y_pred)^2)
    
    Args:
        Y_true: one-hot ground truth (m, 10)
        Y_pred: network output      (m, 10)
    Returns:
        scalar MSE value
    """
    m = Y_true.shape[0]
    return np.sum((Y_true - Y_pred) ** 2) / m

## Step 4 — Backpropagation (Task 3)

In [ ]:
def backpropagation(X, Y_true, Z1, A1, Z2, A2, Z3, A3, W1, W2, W3):
    """
    Backpropagation using the delta (error-signal) method.
    Mirrors the hand calculations in Task 3B exactly.
    
    Returns:
        dW1, db1, dW2, db2, dW3, db3 — gradients for each parameter
    """
    m = X.shape[0]

    # ── Output layer delta ──────────────────────────────────────
    # delta3 = dL/dyhat * sigma'(z3)  =  -2*(Y - A3) * A3*(1-A3)
    delta3 = -2 * (Y_true - A3) * sigmoid_derivative(A3)  # (m, 10)

    # Gradients W3, b3
    dW3 = (A2.T @ delta3) / m   # (64, 10)
    db3 = np.sum(delta3, axis=0, keepdims=True) / m  # (1, 10)

    # ── Hidden layer 2 delta ────────────────────────────────────
    # delta2 = (delta3 @ W3.T) * sigma'(A2)
    delta2 = (delta3 @ W3.T) * sigmoid_derivative(A2)     # (m, 64)

    # Gradients W2, b2
    dW2 = (A1.T @ delta2) / m   # (128, 64)
    db2 = np.sum(delta2, axis=0, keepdims=True) / m  # (1, 64)

    # ── Hidden layer 1 delta ────────────────────────────────────
    # delta1 = (delta2 @ W2.T) * sigma'(A1)
    delta1 = (delta2 @ W2.T) * sigmoid_derivative(A1)     # (m, 128)

    # Gradients W1, b1
    dW1 = (X.T @ delta1) / m    # (784, 128)
    db1 = np.sum(delta1, axis=0, keepdims=True) / m  # (1, 128)

    return dW1, db1, dW2, db2, dW3, db3

## Step 5 — Weight Update (Task 4)

In [ ]:
def update_weights(W1, b1, W2, b2, W3, b3,
                   dW1, db1, dW2, db2, dW3, db3,
                   learning_rate):
    """
    Gradient descent update: W = W - lr * dW
    Returns all updated parameters.
    """
    W1 = W1 - learning_rate * dW1
    b1 = b1 - learning_rate * db1
    W2 = W2 - learning_rate * dW2
    b2 = b2 - learning_rate * db2
    W3 = W3 - learning_rate * dW3
    b3 = b3 - learning_rate * db3
    return W1, b1, W2, b2, W3, b3

## Load MNIST Dataset

In [ ]:
# Download MNIST if not already present
mnist_path = 'data/mnist.npz'
os.makedirs('data', exist_ok=True)
if not os.path.exists(mnist_path):
    url = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz'
    print('Downloading MNIST...')
    urllib.request.urlretrieve(url, mnist_path)
    print('Download complete.')

data = np.load(mnist_path)
X_train = data['x_train'].reshape(-1, 784) / 255.0   # (60000, 784)
Y_train  = data['y_train']                            # (60000,)
X_test   = data['x_test'].reshape(-1, 784)  / 255.0  # (10000, 784)
Y_test   = data['y_test']                             # (10000,)

# One-hot encode labels
def one_hot(y, num_classes=10):
    oh = np.zeros((len(y), num_classes))
    oh[np.arange(len(y)), y] = 1
    return oh

Y_train_oh = one_hot(Y_train)  # (60000, 10)
Y_test_oh  = one_hot(Y_test)   # (10000, 10)

print(f'X_train: {X_train.shape}  Y_train: {Y_train.shape}')
print(f'X_test:  {X_test.shape}   Y_test:  {Y_test.shape}')

## Initialise Weights

In [ ]:
np.random.seed(42)

# Network: 784 → 128 → 64 → 10
# Weights initialised uniformly in [-0.5, 0.5], biases to zero
W1 = np.random.uniform(-0.5, 0.5, (784, 128))
b1 = np.zeros((1, 128))
W2 = np.random.uniform(-0.5, 0.5, (128, 64))
b2 = np.zeros((1, 64))
W3 = np.random.uniform(-0.5, 0.5, (64, 10))
b3 = np.zeros((1, 10))

print('Weights initialised:')
print(f'  W1: {W1.shape}  b1: {b1.shape}')
print(f'  W2: {W2.shape}  b2: {b2.shape}')
print(f'  W3: {W3.shape}  b3: {b3.shape}')

## Training Loop — Mini-Batch Gradient Descent (Task 5)

In [ ]:
learning_rate = 0.1
epochs        = 20
batch_size    = 32
loss_history  = []

for epoch in range(epochs):
    # Shuffle training data each epoch (Task 5 principle)
    idx    = np.random.permutation(X_train.shape[0])
    X_shuf = X_train[idx]
    Y_shuf = Y_train_oh[idx]

    # Mini-batch loop
    for start in range(0, X_train.shape[0], batch_size):
        X_batch = X_shuf[start : start + batch_size]
        Y_batch = Y_shuf[start : start + batch_size]

        # Forward pass
        Z1, A1, Z2, A2, Z3, A3 = forward_pass(X_batch, W1, b1, W2, b2, W3, b3)

        # Backpropagation
        dW1, db1, dW2, db2, dW3, db3 = backpropagation(
            X_batch, Y_batch, Z1, A1, Z2, A2, Z3, A3, W1, W2, W3)

        # Weight update
        W1, b1, W2, b2, W3, b3 = update_weights(
            W1, b1, W2, b2, W3, b3,
            dW1, db1, dW2, db2, dW3, db3,
            learning_rate)

    # Record epoch loss on full training set
    _, _, _, _, _, A3_full = forward_pass(X_train, W1, b1, W2, b2, W3, b3)
    epoch_loss = mse_loss(Y_train_oh, A3_full)
    loss_history.append(epoch_loss)
    print(f'Epoch {epoch+1:>2}/{epochs}  Loss: {epoch_loss:.4f}')

## Required Output 1 — Loss Curve

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, epochs + 1), loss_history, marker='o', linewidth=2, color='steelblue')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Loss Curve — MVC MLP on MNIST')
plt.grid(True, alpha=0.3)
plt.tight_layout()
os.makedirs('report', exist_ok=True)
plt.savefig('report/loss_curve.png', dpi=150)
plt.show()
print(f'Initial loss: {loss_history[0]:.4f}  Final loss: {loss_history[-1]:.4f}')

## Required Output 2 — Test Accuracy

In [ ]:
_, _, _, _, _, A3_test = forward_pass(X_test, W1, b1, W2, b2, W3, b3)
predictions  = np.argmax(A3_test, axis=1)
accuracy     = np.mean(predictions == Y_test) * 100
print(f'Test Accuracy: {accuracy:.2f}%  ({np.sum(predictions == Y_test)}/{len(Y_test)} correct)')

## Required Output 3 — Sample Predictions (one per digit class)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
axes = axes.flatten()

for digit in range(10):
    # Find first test image of this digit class
    idx = np.where(Y_test == digit)[0][0]
    img = X_test[idx].reshape(28, 28)
    pred = predictions[idx]
    true = Y_test[idx]

    axes[digit].imshow(img, cmap='gray')
    color = 'green' if pred == true else 'red'
    axes[digit].set_title(f'True: {true}  Pred: {pred}', color=color, fontsize=10)
    axes[digit].axis('off')

plt.suptitle('Sample Predictions — One per Digit Class (green=correct, red=wrong)',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig('report/sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## Bonus — Confusion Matrix

In [ ]:
# Build confusion matrix manually (no sklearn required)
cm = np.zeros((10, 10), dtype=int)
for t, p in zip(Y_test, predictions):
    cm[t, p] += 1

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel('Predicted label'); ax.set_ylabel('True label')
ax.set_title('Confusion Matrix on Test Set')
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=7)
plt.tight_layout()
plt.savefig('report/confusion_matrix.png', dpi=150)
plt.show()